# pyNTAVIS: three-axis AEDAT files tutorial

This notebook shows how to load and visualise a **three-axis** (X, Y, Z) AEDAT file recorded with the NTAS accelerometer sensor.

In a three-axis file the addresses of each axis are stored in consecutive blocks of `num_channels * (on_off_both + 1)` addresses: first the X axis, then the Y axis and finally the Z axis.

## 0. Setup (Google Colab only)

If you opened this notebook in **Google Colab**, run this cell first. It downloads the pyNTAVIS repository, installs its modified version of `pyNAVIS` (instead of the PyPI release) and moves into the `src/examples` folder so the test files can be found.

When the notebook runs locally (for example with the project's `.venv`), this cell only prints where `pyNAVIS` is loaded from.


In [ ]:
# Setup for Google Colab: use the pyNTAVIS version of pyNAVIS from GitHub, not the PyPI one
import os, sys

if 'google.colab' in sys.modules:
    if not os.path.isdir('/content/pyNTAVIS'):
        !git clone --depth 1 https://github.com/juamonsan/pyNTAVIS.git /content/pyNTAVIS
    !pip uninstall -y -q pyNAVIS
    !pip install -q /content/pyNTAVIS/src
    os.chdir('/content/pyNTAVIS/src/examples')

import pyNAVIS
print('pyNAVIS loaded from:', pyNAVIS.__file__)

## 1. Import the required libraries

`pyNAVIS` is imported from this repository (installed in editable mode in the project's `.venv`). `%matplotlib inline` shows every figure below its code cell.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

from pyNAVIS import Loaders, Plots, Functions, MainSettings

## 2. Create the `MainSettings` object

The parameters must match the test file `NTAS_Accel_3Axxis_43ch_ONOFF_addr4b_ts1.aedat`:

| Parameter | Value | Reason |
|---|---|---|
| `num_channels` | 43 | Each axis has 42 active channels placed in a 43-channel slot (the last one is always empty) |
| `mono_stereo` | `MainSettings.THREE_AXIS` (= 2) | Three sources: X, Y and Z axes |
| `on_off_both` | 1 | The file contains both ON and OFF events |
| `address_size` | 4 | jAER saves 4-byte addresses |
| `ts_tick` | 1 | jAER timestamps have a 1 µs tick |
| `bin_size` | 20000 | 20 000 µs (20 ms) bins for the time-binned plots |

In [ ]:
settings = MainSettings(num_channels=43, mono_stereo=MainSettings.THREE_AXIS, on_off_both=1,
                        address_size=4, ts_tick=1, bin_size=20000)

print('Sources:', settings.source_labels)
print('Addresses per axis:', settings.addresses_per_source)

## 3. Load the AEDAT file

`Loaders.loadAEDAT()` returns a `SpikesFile` object with the addresses and timestamps (in µs) of every spike.

In [ ]:
path = 'test_files/NTAS_Accel_3Axxis_43ch_ONOFF_addr4b_ts1.aedat'
three_axis_file = Loaders.loadAEDAT(path, settings)

print('Number of spikes:', len(three_axis_file.addresses))
print('Duration: {:.3f} s'.format((three_axis_file.max_ts - three_axis_file.min_ts) * 1e-6))

> **Note: the plots show addresses, not channels.**
>
> Each channel produces two addresses: `2·channel` for ON spikes and `2·channel + 1` for OFF spikes. Each axis therefore uses a block of `43 × 2 = 86` addresses, and the full file spans `3 × 86 = 258` addresses (0 to 257):
>
> | Axis | Addresses used | Unused (empty 43rd channel) |
> |---|---|---|
> | X | 0 – 83 | 84 – 85 |
> | Y | 86 – 169 | 170 – 171 |
> | Z | 172 – 255 | 256 – 257 |
>
> To convert an address into its axis, channel and polarity:
>
> - axis = `address // 86` (0 = X, 1 = Y, 2 = Z)
> - channel = `(address % 86) // 2` (0 to 41)
> - polarity = `address % 2` (0 = ON, 1 = OFF)
>
> In the per-axis plots (sections 9 and 10) the addresses of each axis start at 0, so only the last two formulas apply.


## 4. Spikegram

A raster plot where every spike is drawn as a dot: time on the horizontal axis and address on the vertical axis. Each axis (X, Y, Z) is drawn in a different colour.

In [ ]:
Plots.spikegram(three_axis_file, settings, graph_title='Spikegram - 3 axes')
plt.show()

## 5. Sonogram

The spiking activity of every address in bins of `bin_size` µs, shown as a colour map. The X, Y and Z address blocks are labelled on the right side.

In [ ]:
Plots.sonogram(three_axis_file, settings, graph_title='Sonogram - 3 axes')
plt.show()

## 6. Histogram

The total number of spikes per address. The three axes are overlaid on the same per-axis address range (0 to 85) so they can be compared directly.

In [ ]:
spikes_count, _ = Plots.histogram(three_axis_file, settings, graph_title='Histogram - 3 axes')
plt.show()

## 7. Average activity

The number of spikes of each axis in every `bin_size` µs bin, showing how the activity of X, Y and Z evolves over time.

In [ ]:
x_activity, y_activity, z_activity, _ = Plots.average_activity(three_axis_file, settings,
                                                               graph_title='Average activity - 3 axes')
plt.show()

## 8. Extract each axis as a separate file

`Functions.stereo_to_mono()` extracts a single source from a three-axis file (`left_right` = 0, 1, 2 for X, Y, Z). The addresses of each extracted axis start at 0, so they are plotted with mono settings (`mono_stereo=0`).

In [ ]:
axis_settings = MainSettings(num_channels=settings.num_channels, mono_stereo=0, on_off_both=settings.on_off_both,
                             address_size=settings.address_size, ts_tick=settings.ts_tick, bin_size=settings.bin_size)

axis_files = [Functions.stereo_to_mono(three_axis_file, left_right=axis, settings=settings)
              for axis in range(settings.num_sources)]

## 9. Spikegram of each axis

One spikegram per axis.

In [ ]:
for axis_file, label in zip(axis_files, settings.source_labels):
    Plots.spikegram(axis_file, axis_settings, graph_title='Spikegram - ' + label)
plt.show()

## 10. Sonogram of each axis

One sonogram per axis.

In [ ]:
for axis_file, label in zip(axis_files, settings.source_labels):
    Plots.sonogram(axis_file, axis_settings, graph_title='Sonogram - ' + label)
plt.show()